# Run the pilot on a Colab GPU

First: **Runtime -> Change runtime type -> T4 GPU -> Save**.

This notebook runs **phase 2**: the powered fp32 run (3 widths x 5 seeds = 15
training runs, 200 epochs each), about 6 hours on a T4. Results are kept on
Google Drive so a Colab disconnect does not lose progress -- reconnect and
re-run from the top; `phase2.sh` skips any run that already has a
`checkpoints/best.pt`.

Phase 1 (the 30-epoch signal check) is at the bottom if you need to redo it.

In [ ]:
!nvidia-smi -L
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# Clone (or re-verify) the repo, then mount Drive and point results/ at it.
# Self-healing: safe to re-run in any order, e.g. after a disconnect.
%cd /content
![ -d /content/student-teacher-landscape ] || git clone -b dev https://github.com/Iyeba-Kallon/student-teacher-landscape.git
%cd /content/student-teacher-landscape
!git pull --quiet

from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/stl_phase2_results
!rm -rf /content/student-teacher-landscape/results
!ln -s /content/drive/MyDrive/stl_phase2_results /content/student-teacher-landscape/results
!ls -la /content/student-teacher-landscape/  # confirm results/ is a symlink to drive

In [ ]:
# Only extra dependency Colab does not already have. Keep Colab's own torch.
!pip install -q pyhessian
import pyhessian, yaml, numpy, pandas, torchvision  # sanity
print('deps OK')

In [ ]:
# CIFAR-10-C (~2.9 GB, a few minutes). Idempotent. CIFAR-10 auto-downloads later.
!python data/download_cifar10c.py --dest data/

In [ ]:
# Sanity check before spending 6 hours: every fp32 config must read epochs: 200.
!grep -E '^\s+epochs:' configs/teacher_fp32.yaml configs/student_w0.5_fp32.yaml configs/student_w0.25_fp32.yaml

In [ ]:
# Phase 2: 15 training runs -> evaluate -> geometry -> aggregate. ~6h on a T4.
# If this cell disconnects partway, just re-run this cell (after re-running the
# cells above to reconnect Drive) -- finished runs are skipped.
!bash scripts/phase2.sh

In [ ]:
import pandas as pd
df = pd.read_csv('results/pilot_summary.csv')
df[['run_name','mode','width_mult','seed','id_acc','ood_acc_mean','mce_vs_baseline',
    'adaptive_sharpness','hessian_trace','hessian_top_eigenvalue']]

In [ ]:
# Per seed: does each student beat its teacher on OOD, and is it flatter?
t = df[df['mode']=='teacher'].set_index('seed')
for _, s in df[df['mode']=='student'].sort_values(['width_mult','seed']).iterrows():
    ts = t.loc[s['seed']]
    print(f"seed {s['seed']}  w={s['width_mult']:<5} "
          f"OOD {s['ood_acc_mean']:.3f} vs {ts['ood_acc_mean']:.3f} "
          f"({'BEATS' if s['ood_acc_mean']>ts['ood_acc_mean'] else 'below':>5}) | "
          f"sharpness {s['adaptive_sharpness']:.4f} vs {ts['adaptive_sharpness']:.4f} "
          f"({'FLATTER' if s['adaptive_sharpness']<ts['adaptive_sharpness'] else 'sharper'})")

In [ ]:
# Results already live on Drive (stl_phase2_results/), but a zip is handy too.
!zip -qr results_phase2.zip results/ -x 'results/**/checkpoints/*'
from google.colab import files
files.download('results_phase2.zip')

---
## Phase 1 (redo only if needed)

The 30-epoch, 2-seed signal check. Already run once (see phase1-results in
project memory) -- this is here only in case you need to reproduce it.

```bash
!bash scripts/phase1.sh
```